In [ ]:
from matplotlib import pyplot as plt

import matplotlib.image as mimg

import ipywidgets as ipw
import numpy as np

from astropy.nddata import CCDData, block_reduce
from astropy.visualization import AsymmetricPercentileInterval, LogStretch, ManualInterval

from astrowidgets.bqplot import ImageWidget

from photutils.background import Background2D, MedianBackground

In [ ]:
image_widgets = dict(
    red=ImageWidget(),
    green=ImageWidget(),
    blue=ImageWidget()
)

In [ ]:
data_sm = {}
data = {}
sc_raw = {}
sc_raw_f = {}
data_sm_raw = {}
data_raw_unmod = {}
bkgd_sm = {}
bkgd_f = {}

def get_scaled_image_data(viewer, data):
    """
    Get scaled image data from a viewer using its current stretch and cuts settings.

    Parameters
    ----------
    viewer : ImageWidget
        The image viewer containing the current stretch and cuts configuration.
    data : numpy.ndarray
        The image data to scale.

    Returns
    -------
    numpy.ndarray
        The scaled image data after applying the viewer's stretch and cuts.
    """
    return viewer._get_stretch()(viewer.cuts(data))

In [ ]:
def make_image_load_observer(stretch):
    """
    Create an observer function that loads image data for red, green, and blue channels.

    Parameters
    ----------
    stretch : ipywidgets.Dropdown
        The shared stretch chooser widget whose value is applied to all image viewers.

    Returns
    -------
    callable
        An observer function that accepts a change dict, loads FITS images from disk,
        applies block reduction, computes backgrounds, and populates the global data dicts.
    """
    def load_image_data(change):
        """
        Load block-reduced image data, compute backgrounds, and apply subtraction
        based on the current state of subtract_bkgd_checkbox.

        Parameters
        ----------
        change : dict
            Widget change dictionary with key 'new' containing the object name string.
        """
        global object_name
        obj = change["new"]
        red = CCDData.read(f'combined/combined_light_filter_rp.fit')
        greenish = CCDData.read(f'combined/combined_light_filter_V.fit')
        blue = CCDData.read(f'combined/combined_light_filter_B.fit')

        reduce_fac = 8
        reduce_func = np.mean
        red_sm = block_reduce(red.data, reduce_fac, func=reduce_func)
        green_sm = block_reduce(greenish.data, reduce_fac, func=reduce_func)
        blue_sm = block_reduce(blue.data, reduce_fac, func=reduce_func)

        # Compute backgrounds for small images; store raw data and backgrounds separately
        for sm_image, color in zip([red_sm, green_sm, blue_sm], ['red', 'green', 'blue']):
            bkgd = Background2D(sm_image, (64, 64), filter_size=(3, 3), bkg_estimator=MedianBackground())
            bkgd_sm[color] = bkgd.background
            data_sm_raw[color] = sm_image.copy()

        # Compute backgrounds for full images; copy first to avoid modifying CCDData in place
        for raw_array, color in zip([red.data, greenish.data, blue.data], ['red', 'green', 'blue']):
            raw_copy = raw_array.copy()
            bkgd = Background2D(raw_copy, (512, 512), filter_size=(3, 3), bkg_estimator=MedianBackground())
            bkgd_f[color] = bkgd.background
            data_raw_unmod[color] = raw_copy

        # Apply (or skip) background subtraction based on checkbox
        subtract = subtract_bkgd_checkbox.value
        for color in ['red', 'green', 'blue']:
            if subtract:
                data_sm[color] = data_sm_raw[color] - bkgd_sm[color]
                data[color] = data_raw_unmod[color] - bkgd_f[color]
            else:
                data_sm[color] = data_sm_raw[color].copy()
                data[color] = data_raw_unmod[color].copy()

        image_widgets['red'].load_array(data_sm['red'])
        image_widgets['green'].load_array(data_sm['green'])
        image_widgets['blue'].load_array(data_sm['blue'])
        for color in ['red', 'green', 'blue']:
            image_widgets[color].stretch = stretch.value

        object_name = blue.header['OBJECT']

    return load_image_data

In [ ]:
def make_slider():
    """
    Create a float range slider widget for setting black and white pixel levels.

    Returns
    -------
    ipywidgets.FloatRangeSlider
        A slider with range 0 to 100000/64, step 100/64, and full-width layout.
    """
    slider = ipw.FloatRangeSlider(min=0, max=100000/64, step=100/64,
                                  description='Set black and white',
                                  style={'description_width': 'initial'},
                                  continuous_update=False,
                                  layout={'width': '100%'}
                                 )
    return slider

In [ ]:
def make_strech_chooser():
    """
    Create a dropdown widget for selecting the image stretch type.

    Returns
    -------
    ipywidgets.Dropdown
        A dropdown widget with options 'linear', 'log' and 'sqrt'.
    """
    chooser = ipw.Dropdown(options=["linear", "log", "sqrt"], description="Stretch")
    return chooser

In [ ]:
level_sliders = dict(
    red=make_slider(),
    green=make_slider(),
    blue=make_slider()
)

stretch_chooser = make_strech_chooser()

subtract_bkgd_checkbox = ipw.Checkbox(
    value=True,
    description='Subtract background',
    style={'description_width': 'initial'},
)

In [ ]:
def make_observer(color):
    """
    Create a slider observer function for a given color channel.

    Parameters
    ----------
    color : str
        The color channel to observe ('red', 'green', or 'blue').

    Returns
    -------
    callable
        An observer function that updates the image viewer's cuts and recomputes
        scaled image data for both preview and full-resolution when the slider changes.
    """
    def observer(change):
        """
        Update image cuts and recompute scaled data for the color channel.

        Parameters
        ----------
        change : dict
            Widget change dictionary with key 'new' containing a (min, max) tuple
            representing the new black and white level values.
        """
        minval, maxval = change['new']
        image_widgets[color].cuts = ManualInterval(minval, maxval)
        sc_raw[color] = get_scaled_image_data(image_widgets[color], data_sm[color])
        sc_raw_f[color] = get_scaled_image_data(image_widgets[color], data[color])

    return observer

def make_stretch_observer():
    """
    Create an observer that applies the shared stretch setting to all color channels.

    Returns
    -------
    callable
        An observer function that updates the stretch and recomputes scaled image data
        for all three color channels when the shared stretch dropdown changes.
    """
    def observer(change):
        """
        Update stretch and recompute scaled data for all color channels.

        Parameters
        ----------
        change : dict
            Widget change dictionary with key 'new' containing the stretch type string
            ('linear' or 'log').
        """
        for color in ['red', 'green', 'blue']:
            image_widgets[color].stretch = change['new']
            sc_raw[color] = get_scaled_image_data(image_widgets[color], data_sm[color])
            sc_raw_f[color] = get_scaled_image_data(image_widgets[color], data[color])
    return observer

load_obs = make_image_load_observer(stretch_chooser)

load_obs({"new": "dummy"})

## 1. Adjust each of the combined image (red, green, blue) so that the background is black and you can see the detail you want

In [ ]:
tab_set = ipw.Tab()
kids = []
boxes = {}
colors = ['red', 'green', 'blue']
object_label = ipw.HTML("<h3>Object name:</h3>")
object_name = ipw.Text(description="", value=object_name)

obbie = ipw.HBox()
obbie.children = [object_label, object_name]

vb = ipw.VBox(layout={'width': '100%'})

for idx, color in enumerate(colors):
    boxes[color] = ipw.VBox(children=[level_sliders[color], image_widgets[color]])
    this_observer = make_observer(color)
    level_sliders[color].observe(this_observer, names='value')
    this_observer(dict(new=level_sliders[color].value))
    kids.append(boxes[color])

stretch_obs = make_stretch_observer()
stretch_chooser.observe(stretch_obs, names='value')
stretch_obs(dict(new=stretch_chooser.value))

tab_set.children = kids
tab_set.titles = colors
vb.children = [ipw.HBox([subtract_bkgd_checkbox, stretch_chooser]), tab_set]


In [ ]:
def rgb_scaling(sc_data, r=0.5, g=0.5, b=0.5):
    """
    Scale and combine RGB channel data into a color image.

    Parameters
    ----------
    sc_data : dict
        Dictionary with 'red', 'green', 'blue' keys containing scaled 2D arrays.
    r, g, b : float, optional
        Scaling factors for each channel. Default is 0.5.

    Returns
    -------
    numpy.ndarray
        3D array with shape (*image_shape, 3), clipped to [0, 1].
    list
        Per-channel max values before clipping.
    """
    red_sc = r * sc_data["red"]
    green_sc = g * sc_data["green"]
    blue_sc = b * sc_data["blue"]
    comb = np.zeros(list(red_sc.shape) + [3])
    comb[:, :, 0] = red_sc
    comb[:, :, 1] = green_sc
    comb[:, :, 2] = blue_sc
    maxes = [np.nanmax(red_sc), np.nanmax(green_sc), np.nanmax(blue_sc)]
    comb = 2 * comb
    comb[comb > 1] = 1.0
    return comb, maxes


def quick_color_rgb(r=0.5, g=0.5, b=0.5, output=None):
    """
    Compose and display a quick-look color RGB image from scaled channel data.

    Parameters
    ----------
    r, g, b : float, optional
        Scaling factors for each channel. Default is 0.5.
    """
    comb, maxes = rgb_scaling(sc_raw, r, g, b)
    max_img = np.nanmax(comb.flatten())
    if output is None:
        return
    with output:
        output.clear_output(wait=True)
        #print("MEEEEEP", r, g, b)
        fig, ax = plt.subplots(figsize=(8, 8))

        ax.set_title(f'{max_img=:.3f} {r=:.2f} {g=:.2f} {b=:.2f}\n{maxes=}')
        ax.tick_params(labelbottom=False, labelleft=False, labelright=False, labeltop=False)
        ax.imshow(comb, vmin=0, vmax=1)
        #plt.ion()
        plt.show()

        #plt.ioff()
        #display(fig)
        #plt.close(fig)

## Adjust the contribution of the red, green and blue images to the final image

In [ ]:
r_slider = ipw.FloatSlider(value=0.5, min=0, max=1, step=0.01, description='Red',
                           style={'description_width': 'initial'}, layout={'width': '100%'})
g_slider = ipw.FloatSlider(value=0.5, min=0, max=1, step=0.01, description='Green',
                           style={'description_width': 'initial'}, layout={'width': '100%'})
b_slider = ipw.FloatSlider(value=0.5, min=0, max=1, step=0.01, description='Blue',
                           style={'description_width': 'initial'}, layout={'width': '100%'})

preview_output = ipw.Output()

rgb_mixer = ipw.VBox([r_slider, g_slider, b_slider, preview_output], layout=dict(width="90%"))

def update_preview(change):
    """
    Refresh the preview image when any RGB slider value changes.

    Parameters
    ----------
    change : dict
        Widget change dictionary passed by the observer framework (unused directly;
        current slider values are read from the slider widgets).
    """
    quick_color_rgb(r_slider.value, g_slider.value, b_slider.value, output=preview_output)

for slider in [r_slider, g_slider, b_slider]:
    slider.observe(update_preview, names='value')

# Re-render the preview whenever vb's level sliders or stretch dropdown changes
for color in colors:
    level_sliders[color].observe(update_preview, names='value')

stretch_chooser.observe(update_preview, names='value')

def on_subtract_change(change):
    """
    Apply or remove background subtraction when the checkbox is toggled.

    Updates data_sm and data for all channels, reloads the image viewers,
    recomputes sc_raw/sc_raw_f, and refreshes the color preview.
    """
    if not data_sm_raw:
        return  # images not loaded yet
    subtract = change['new']
    for color in ['red', 'green', 'blue']:
        if subtract:
            data_sm[color] = data_sm_raw[color] - bkgd_sm[color]
            data[color] = data_raw_unmod[color] - bkgd_f[color]
        else:
            data_sm[color] = data_sm_raw[color].copy()
            data[color] = data_raw_unmod[color].copy()
        image_widgets[color].load_array(data_sm[color])
        image_widgets[color].stretch = stretch_chooser.value
        make_observer(color)(dict(new=level_sliders[color].value))
    update_preview(None)

subtract_bkgd_checkbox.observe(on_subtract_change, names='value')

In [ ]:
def full_res_color_rgb(r=0.5, g=0.5, b=0.5):
    """
    Compose and display a full-resolution color RGB image from scaled channel data.

    Parameters
    ----------
    r, g, b : float, optional
        Scaling factors for each channel. Default is 0.5.
    """
    comb, maxes = rgb_scaling(sc_raw_f, r, g, b)
    fig, ax = plt.subplots(figsize=(20, 20))
    max_img = np.nanmax(comb.flatten())
    min_img = np.nanmin(comb.flatten())
    plt.title(f'{max_img=:.3f} {min_img=:.3f} {r=:.2f} {g=:.2f} {b=:.2f}\n{maxes=}')
    ax.tick_params(labelbottom=False, labelleft=False, labelright=False, labeltop=False)
    ax.imshow(comb, vmin=0, vmax=1)
    display(fig)
    plt.close(fig)

In [ ]:
def save_widgets():
    """
    Build and return a widget UI and refresh function for the full-resolution save tab.

    The refresh function shows a status message while generating the full-resolution
    image preview, then replaces it with the rendered image. The save button writes
    the raw RGB array to disk via mimg.imsave (no figure decorations).

    Returns
    -------
    tuple[ipywidgets.VBox, callable]
        The save tab widget and a refresh() function to regenerate the image preview.
    """
    filename_input = ipw.Text(
        description='Add to filename:',
        value='',
        placeholder='e.g. _v2',
        style={'description_width': 'initial'},
        layout={'width': '400px'}
    )

    status_html = ipw.HTML('')
    full_res_output = ipw.Output()
    save_button = ipw.Button(description='Save image', button_style='success')
    save_status_label = ipw.Label('')

    def refresh():
        status_html.value = '<p style="padding:10px 0">Generating full resolution image…</p>'
        with full_res_output:
            full_res_output.clear_output()
            full_res_color_rgb(r_slider.value, g_slider.value, b_slider.value)
        status_html.value = ''

    def on_save(b):
        suffix = filename_input.value
        filename = f'full_res_color_{object_name.value}{suffix}.png'
        comb, _ = rgb_scaling(sc_raw_f, r_slider.value, g_slider.value, b_slider.value)
        mimg.imsave(filename, comb)
        save_status_label.value = f'Saved: {filename}'

    save_button.on_click(on_save)

    widget = ipw.VBox([
        filename_input,
        ipw.HBox([save_button, save_status_label]),
        status_html,
        full_res_output,
    ])

    return widget, refresh

In [ ]:
save_widget, refresh_save = save_widgets()

tabbie = ipw.Tab()
tabbie.children = [vb, rgb_mixer, save_widget]
tabbie.titles = ['1. Adjust B/W', '2. Adjust colors', '3. Save']

def on_tab_change(change):
    if change['new'] == 2:
        refresh_save()

tabbie.observe(on_tab_change, names='selected_index')
tabbie